# Info from Canvas


## Viterbi Algorithm for Hidden Markov Models
Your group will demonstrate your understanding of Hidden Markov Models by implementing the Viterbi algorithm for finding the most likely sequence of hidden states given a sequence of observations.

The Viterbi algorithm represents a dynamic programming approach to decoding hidden states, offering several advantages over naive approaches:

### Key Features

Optimal Path Finding: Identifies the most probable sequence of hidden states
Dynamic Programming: Uses efficient tabulation to avoid redundant calculations
Log-Space Computation: Prevents numerical underflow with large sequences
Traceback Mechanism: Reconstructs the optimal state path after computation

### Algorithm Structure

The Viterbi algorithm involves these key steps:

1. Initialization:
Set up a probability matrix using initial probabilities and the first observation
Initialize traceback matrix for path reconstruction
2. Recursion:
For each position and possible state, calculate the maximum probability
Store both probabilities and traceback pointers
Apply transition and emission probabilities at each step
3. Termination:
Identify the final state with the highest probability
Trace back through the matrix to reconstruct the optimal path

### Applications Beyond Sequence Analysis

While primarily used for biological sequence analysis, this approach has broader applications:

- Speech Recognition: Identifying phonemes in audio signals
- Part-of-Speech Tagging: Determining grammatical roles in text
- Gene Finding: Locating coding regions in DNA sequences
- Financial Modeling: Detecting market regimes in time series data

### Computational Considerations

Important factors to consider in implementation:

Time Complexity: O(N x K^2) where N is sequence length and K is number of states
Space Complexity: O(N x K) for storing the dynamic programming matrix
Numerical Stability: Using log probabilities to prevent underflow
Edge Cases: Handling zero probabilities with pseudocounts

## Forward, Backward, and Forward-Backward Algorithms for Hidden Markov Models
Your group will demonstrate your understanding of Hidden Markov Models by implementing the Forward, Backward, and Forward-Backward algorithms for calculating sequence likelihoods and posterior probabilities.

These algorithms represent fundamental approaches to HMM probability inference, offering several advantages over naive enumeration:

### Key Features

- **Joint Probability Calculation**: The Forward algorithm computes *P(x1, ..., xt, qt = i)*
, the joint probability of the partial observation sequence and ending in state *i*
- **Conditional Future Probability**: The Backward algorithm computes *P(xt+1, ..., xn | qt = i)*
, the probability of future observations given the current state
- **Posterior Marginals**: The Forward-Backward algorithm combines these to determine *P(qt = i | x1, ..., xn)*
, the probability of being in each state at each position given the entire observation sequence
- **Dynamic Programming**: Uses efficient recursion to avoid exponential complexity

**Algorithm Structure**

The implementation involves these key components:

1. **Forward Algorithm**:
    - Initialize with initial probabilities and the first observation
    - Recursively calculate joint probabilities by summing over all possible previous states
    - Compute the total sequence probability by summing the final column
2. **Backward Algorithm**:
    - Initialize the final position with a value of 1 for all states
    - Recursively calculate conditional probabilities, working from the end to the beginning
    - Combine with initial probabilities to get sequence likelihood
3. **Forward-Backward Algorithm**:
    - Calculate both forward and backward matrices
    - Combine to compute the posterior marginal probability for each state at each position
    - Normalize to ensure a valid probability distribution

**Applications Beyond Sequence Analysis**

While primarily used for biological sequence analysis, these approaches have broader applications:

- **Protein Structure Prediction**: Calculating the likelihood of secondary structures
- **Financial Time Series**: Detecting hidden economic states
- **Natural Language Processing**: Word sense disambiguation
- **Gesture Recognition**: Identifying patterns in motion data

### Computational Considerations

- **Time Complexity**: O(N x K^2)
where 
 N is the sequence length and 
 K is the number of states
- **Space Efficiency**: Matrices can be optimized for memory usage
- **Numerical Stability**: Preventing underflow with scaling techniques
- **Validation**: Forward and Backward should yield nearly identical sequence probabilities

# Thoughts and considerations

Define HMM class with attributes: 
    - initial probabilities -> Dict 
    - transition probabilities -> Dict of Dicts 
    - emission probabilities -> Dict of Dicts
Track states and potential emissions from given observations

*For viterbi, assumption is all emissions and states are defined... but for future algorithms (potentially) consider ability to add and track emissions/states. 

1) initialization 
    - for each observations, calculate probability that observation occurs in each state given transition probabilities and most recent probability calculated
            - most recent probability * transition probability * emission probability in state.
            - For each state in state tracker, the above calc is done





# Pseudocode

Class HMM(emission_prob, transition_prob, initial_prob):

    FUNCTION viterbi(self, observations):
    
    INITIALIZATION: start with initialization of probabilities for each state for each observation
    initialize two matrices, probability matrix and move matrix 
    first column of probability matrix:
        for each state: 
            initial probability of state k * emission probability for observation 0 in state k

    RECURSION: 
        For each remaining observation: 
            for each possible state:
                previous probability for preceding state  * transition probability * emission probability
                take max value after all states are iterated through
                store state that contributed to max value
                return score matrix, move matrix, max of last column (or Termination function)
                
    TERMINATION:
        after all observations are iterated through
        Take max from last column and begin traceback
    
    TRACEBACK:
        take state contributing to max of last column
        add to state path list 
        move one back in the matrix and look at cell in corresponding state row
        get state contributing to max in new cell
        add to state path list
        move one back in matrix and look at cell in corresponding state row
        repeat for all observations
        reverse list of state path
        return state path list (final output)

    return output from traceback

    FUNCTION forward(self, observations):

        INITIALIZATION: start with initialization of probabilities for each state for each observation
        initialize probability matrix
        first column of probability matrix:
            for each state: 
                initial probability of state k * emission probability for observation 0 in state k

        RECURSION: 
            For each remaining observation: 
                for each possible state:
                    for each possible previous state:
                        previous probability for preceding state  * transition probability * emission probability
                        add value to cumulative score counter (np.logaddexp)
                    store cumulative score for that cell in matrix, reset cumulative score
            return matrix

        TERMINATION:
            after all observations are iterated through
            sum values from last column --> overall probability

        return matrix and overall probability

    FUNCTION backward(self, observations):

        reverse order of observations
        call forward algorithm with reversed observations
        return matrix and overall probability

    FUNCTION forward_backward(self, observations, index):

        call forward algorithm
        call backward algorithm
        calculate average overall probability from forward and backward
        for each state:
            access appropriate index at forward and backward matrices
            calculate marginal probability for state (fk + bk - overall_probability)
        return marginal probabilities

In [9]:
import numpy as np

In [10]:
class HMM:
    """
    Class HMM implements a Hidden Markov Model with Viterbi decoding.

    Attributes:
        - initial_prob (dict): dictionary of {state: probability} for initial state probabilities
        - transition_prob (dict): dictionary of dictionaries of {state: {state: probability}} for transition probabilities
        - emission_prob (dict): dictionary of dictionaries of {state: {emission: probability}} for emission probabilities
        - states (list): list of state names derived from initial_prob keys
        - emissions (list): list of emission names derived from emission_prob keys
    Methods:
        - _initialization: Initializes the probability and traceback matrices for the Viterbi algorithm
        - _recursion: Fills in the probability and traceback matrices using dynamic programming
        - _traceback: Traces back through the move matrix to recover the optimal path
        - viterbi: Runs the full Viterbi algorithm and returns the most probable state sequence
    """

    def __init__(self, initial_prob, transition_prob, emission_prob):

        self.initial_prob = initial_prob
        self.transition_prob = transition_prob
        self.emission_prob = emission_prob
        self.states = list(initial_prob.keys())
        self.emissions = list(emission_prob[self.states[0]].keys())

    def _initialization(self, observations):
        """
        Initializes the probability and traceback matrices for the Viterbi algorithm.

        Parameters:
            - observations (list): list of observed emissions
        Returns:
            - probability_matrix (np.ndarray): (states x observations) matrix of log probabilities with column 0 filled with log(initial_prob) + log(emission_prob) for each state
            - move_matrix (np.ndarray): (states x observations) traceback matrix with column 0 initialized to each state's own index
        """

        # Initialize probability and traceback matrices
        probability_matrix = np.zeros((self.rows, self.columns))
        move_matrix = np.zeros((self.rows, self.columns))

        # For each possible state, calculate initial_prob * emission 0 at that state
        for state in range(self.rows):
            probability_matrix[state, 0] = np.log(
                self.initial_prob[self.state_index[state]]
            ) + np.log(self.emission_prob[self.state_index[state]][observations[0]])
            move_matrix[state, 0] = state

        return probability_matrix, move_matrix

    def _recursion(self, observations, probability_matrix, move_matrix):
        """
        Fills in the probability and traceback matrices using dynamic programming.

        For each observation and state, computes the maximum log probability over all
        possible previous states using: prev_prob + log(transition_prob) + log(emission_prob).

        Parameters:
            - observations (list): list of observed emissions
            - probability_matrix (np.ndarray): initialized (states x observations) log probability matrix
            - move_matrix (np.ndarray): initialized (states x observations) traceback matrix
        Returns:
            - probability_matrix (np.ndarray): fully filled log probability matrix
            - move_matrix (np.ndarray): fully filled traceback matrix, where each cell stores
              the index of the previous state that yielded the maximum probability
        """

        # Iterate through each observation
        for obs in range(1, self.columns):

            # For each state, find max joint probability
            for state in range(self.rows):

                # Initialize max value trackers
                max_val = -np.inf
                max_obs = 0

                # For each potentially previous state calculate the joint probability of current state
                for prev_state in range(self.rows):
                    prev_prob = probability_matrix[prev_state, obs - 1]
                    trans_prob = np.log(
                        self.transition_prob[self.state_index[prev_state]][
                            self.state_index[state]
                        ]
                    )
                    emission_prob = np.log(
                        self.emission_prob[self.state_index[state]][observations[obs]]
                    )
                    curr_val = prev_prob + trans_prob + emission_prob

                    # If calculated probability > max value --> update
                    if curr_val > max_val:
                        max_val = curr_val
                        max_obs = prev_state

                # Update probability and move matrices with max value and state we transitioned from
                probability_matrix[state, obs] = max_val
                move_matrix[state, obs] = max_obs

        return probability_matrix, move_matrix

    def _traceback(self, move_matrix, best_path_start):
        """
        Traces back through the move matrix to recover the optimal state sequence.

        Starting from the best final state, follows the traceback moves stored in
        move_matrix from right to left to reconstruct the most probable path.

        Parameters:
            - move_matrix (np.ndarray): fully filled (states x observations) traceback matrix
            - best_path_start (int): index of the state with the highest log probability
              in the final column of the probability matrix
        Returns:
            - best_path (list): ordered list of state names representing the most probable
              state sequence for the given observations
        """

        # Create list to track states from best path
        best_state = best_path_start
        best_path = [self.state_index[best_state]]

        # For each column in the matrix, walk back from the last column finding the best state (max)
        for i in reversed(
            range(2, self.columns + 1)
        ):  # Ignore first column as value is initial state
            best_state = int(move_matrix[best_state, i - 1])

            # Append appropriate state name to best path list
            best_path.append(self.state_index[best_state])

        # Return reverse best path
        return best_path[::-1]

    def viterbi(self, observations):
        """
        Runs the full Viterbi algorithm and returns the most probable state sequence.

        Executes initialization, recursion, termination, and traceback steps
        to find the most probable hidden state sequence for a given observation sequence.

        Parameters:
            - observations (list): list of observed emissions
        Returns:
            - best_path (list): ordered list of state names representing the most probable
              state sequence for the given observations
        """

        # Create rows and columns variables
        self.rows = len(self.states)
        self.columns = len(observations)

        # Create index to track what row each state is {0: state1, 1: state2, ...}
        self.state_index = {i: state for i, state in enumerate(self.states)}

        # Initialization step
        probability_matrix, move_matrix = self._initialization(observations)

        # Recursion step
        probability_matrix, move_matrix = self._recursion(
            observations, probability_matrix, move_matrix
        )
        print(probability_matrix)
        print(move_matrix)

        # Termination step --> get row of max value
        best_path_start = int(probability_matrix[:, -1].argmax())

        # Traceback step
        best_path = self._traceback(move_matrix, best_path_start)

        return best_path

    def forward(self, observations):
        """
        Runs the forward algorithm and returns the probability matrix and overall sequence probability.

        Executes initialization and matrix calculation steps to compute the log probability
        of each state at each position, then sums the final column to get the overall
        log probability of the observation sequence.

        Parameters:
            - observations (list): list of observed emissions
        Returns:
            - probability_matrix (np.ndarray): (states x observations) matrix of log probabilities,
            where each cell represents the log probability of being in a given state
            at a given position having seen all previous observations
            - overall_probability (float): log probability of the full observation sequence,
            computed as the log-sum-exp of the final column of the probability matrix
        """

        # Create rows and columns variables
        self.rows = len(self.states)
        self.columns = len(observations)

        # Create index to track what row each state is {0: state1, 1: state2, ...}
        self.state_index = {i: state for i, state in enumerate(self.states)}

        # Initialization step
        probability_matrix, _ = self._initialization(observations)

        # Calculate probability matrix using observations
        probability_matrix = self._calculate_matrix(probability_matrix, observations)

        # Calculate overall probability (sum of last column)
        overall_probability = np.logaddexp.reduce(probability_matrix[:, -1])

        return probability_matrix, overall_probability

    def _calculate_matrix(self, probability_matrix, observations):
        """
        Fills in the probability matrix using the forward algorithm recurrence.

        For each observation and state, computes the cumulative log probability by summing
        over all possible previous states using: log-sum-exp(prev_prob + log(transition_prob)
        + log(emission_prob)).

        Parameters:
            - probability_matrix (np.ndarray): initialized (states x observations) log probability matrix
            - observations (list): A list of observed emissions
        Returns:
            - probability_matrix (np.ndarray): fully filled log probability matrix, where each cell
            stores the log of the total probability of reaching that state at that position
            having seen all previous observations
        """

        # Iterate through each observation
        for obs in range(1, self.columns):

            # For each state, find max joint probability
            for state in range(self.rows):

                # Initialize cumulative probabilty
                cumulative_prob = -np.inf

                # For each potentially previous state calculate the joint probability of current state
                for prev_state in range(self.rows):
                    prev_prob = probability_matrix[prev_state, obs - 1]
                    trans_prob = np.log(
                        self.transition_prob[self.state_index[prev_state]][
                            self.state_index[state]
                        ]
                    )
                    emission_prob = np.log(
                        self.emission_prob[self.state_index[state]][observations[obs]]
                    )
                    curr_val = prev_prob + trans_prob + emission_prob

                    # Add calculated probability to cumulative probability counter
                    cumulative_prob = float(np.logaddexp(cumulative_prob, curr_val))

                # Update matrix with calculated probability for that state
                probability_matrix[state, obs] = cumulative_prob

        return probability_matrix

    def backward(self, observations):
        """
        Runs the backward algorithm by applying the forward algorithm on the reversed observation sequence.

        Parameters:
            - observations (list): list of observed emissions
        Returns:
            - backwards_prob_matrix (np.ndarray): (states x observations) matrix of log probabilities
            computed on the reversed observation sequence
            - backwards_overall_probability (float): log probability of the reversed observation
            sequence, equivalent to the overall log probability of the original sequence
        """

        # Reverse observations
        reverse_obs = observations[::-1]

        # Run foward algorithm on reversed observations
        backwards_prob_matrix, backwards_overall_probability = self.forward(reverse_obs)

        return backwards_prob_matrix, backwards_overall_probability

    def forward_backward(self, observations, index):
        """
        Computes the marginal posterior probability of each state at a given position.

        Combines the forward and backward log probabilities at a given index to estimate
        the probability of being in each state at that position given the full observation
        sequence. Probabilities are computed in log space and returned as a dictionary.

        Parameters:
            - observations (list): list of observed emissions
            - index (int): position in the observation sequence for which to compute
            marginal posterior probabilities
        Returns:
            - marg_probs (dict): dictionary of {state: marginal_posterior_probability} where
            each value is the marginal posterior log probability of being in that state
            at the given index, computed as fk + bk - average_probability
        """

        # Call forward algorithm
        forward_matrix, forward_probability = self.forward(observations)

        # Call backward algorithm
        backwards_matrix, backwards_probability = self.backward(observations)

        # Calculate average overall probability
        average_probability = (forward_probability + backwards_probability) / 2

        # Initialize marginal probabilities dictionary
        marg_probs = {}

        # Iterate through each possible state
        for row, state in self.state_index.items():

            # Access forward matrix at position index --> fk
            fk = forward_matrix[row, index]

            # Access backward matrix at position (len(observations) - index - 1) --> bk
            bk = backwards_matrix[row, len(observations) - index - 1]

            # Calculate marginal posterior probability (fk * bk / overall prob)
            marginal_posterior = (
                fk + bk - average_probability
            )  # log space so add and subtract
            marg_probs[state] = float(marginal_posterior)

        return marg_probs

# Examples

In [11]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {"I": 0.2, "G": 0.8}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {"I": {"I": 0.7, "G": 0.3}, "G": {"I": 0.1, "G": 0.9}}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
}

In [12]:
hmm = HMM(init_probs, trans_probs, emit_probs)
hmm.viterbi(obs)

[[ -2.52572864  -3.79869432  -5.07166     -7.73092003  -9.00388571
  -11.66314575 -12.81451921 -15.47377925 -17.22494532]
 [ -1.83258146  -3.54737989  -5.26217832  -6.57151164  -8.28631007
   -9.59564339 -11.31044182 -12.61977514 -13.92910846]]
[[0. 0. 0. 0. 0. 0. 1. 0. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]]


['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']

In [13]:
hmm.forward_backward(obs, 4)

-12.6367267220508 -12.512628651893948


{'I': -3.743625771414358, 'G': -2.1850520829008477}

In [14]:
# Example observation sequence
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {"I": 0.1, "G": 0.9}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {"I": {"I": 0.6, "G": 0.4}, "G": {"I": 0.1, "G": 0.9}}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4},
}

In [15]:
hmm = HMM(init_probs, trans_probs, emit_probs)
hmm.viterbi(obs)

[[ -4.60517019  -4.24052707  -5.66764343  -7.09475978  -8.52187614
  -11.33528686 -14.14869757 -14.59498468]
 [ -1.02165125  -3.42959686  -5.83754246  -8.24548807 -10.31363561
  -10.3544576  -11.37610885 -13.78405446]]
[[0. 1. 0. 0. 0. 0. 0. 1.]
 [1. 1. 1. 1. 0. 0. 1. 1.]]


['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']

In [16]:
hmm.forward_backward(obs, 3)

-12.08581295976733 -12.2038533415281


{'I': -2.8017891169975453, 'G': -4.195216905339594}